In [25]:
# %% [Cell 1: Setup & Imports]
import os
import time
import pandas as pd
import numpy as np
import yfinance as yf
import torch
import torch.nn.functional as F
from newsapi import NewsApiClient
from datetime import datetime, timedelta
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import tensorflow as tf
from tensorflow.keras import layers, models

# Suppress Warnings
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
pd.options.mode.chained_assignment = None

In [26]:
# %% [Cell 2: Initialize FinBERT]
# This loads the model into memory once.
print("Loading FinBERT Model...")
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
finbert_model = AutoModelForSequenceClassification.from_pretrained(model_name)

def get_sentiment(text):
    """Processes text and returns a score between -1 and 1."""
    if not text or pd.isna(text): return 0
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = finbert_model(**inputs)
    probs = F.softmax(outputs.logits, dim=-1).numpy()[0]
    # FinBERT labels: [Positive, Negative, Neutral]
    return probs[0] - probs[1]

Loading FinBERT Model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 44771.91it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
# %% [Cell 3: Data Acquisition & Feature Engineering - FINAL VERSION]
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import time

# 1. Setup
end_date = datetime.now()
start_date = end_date - timedelta(days=27)
from_str = start_date.strftime('%Y-%m-%d')
today_str = end_date.strftime('%Y-%m-%d')

print(f"Fetching data starting from {from_str}...")

processed_news = []

# --- PART A: Improved News Fetching ---
# Combining historical and live into one logic stream
queries = ['Amazon AND stock', 'Amazon AND earnings', 'AMZN stock']

for q_string in queries:
    try:
        # Fetching top 100 to ensure we get depth across the date range
        batch = newsapi.get_everything(
            q=q_string,
            from_param=from_str,
            language='en',
            sort_by='relevancy', # Changed to relevancy to get "bigger" stories across the month
            page_size=100
        )
        articles = batch.get('articles', [])
        for art in articles:
            processed_news.append({
                'Date': pd.to_datetime(art['publishedAt']).date(), # Normalize to Date only
                'Sentiment': get_sentiment(art['title'])
            })
        print(f"Fetched {len(articles)} articles for query: {q_string}")
    except Exception as e:
        print(f"Error fetching '{q_string}': {e}")

# 2. Aggregate Sentiment
sentiment_df = pd.DataFrame(processed_news)
if not sentiment_df.empty:
    sentiment_df = sentiment_df.groupby('Date').agg(
        Sentiment_Score=('Sentiment', 'mean'),
        Article_Count=('Sentiment', 'count')
    ).reset_index()
    sentiment_df['Date'] = pd.to_datetime(sentiment_df['Date'])
else:
    print("⚠️ No news articles found! Check your API key or query.")

# 3. Fetch Prices
prices = yf.download(ticker_symbol, start=from_str)
if isinstance(prices.columns, pd.MultiIndex):
    prices.columns = prices.columns.get_level_values(0)
prices = prices[['Close']].reset_index()
prices['Return'] = prices['Close'].pct_change()
prices['Date'] = pd.to_datetime(prices['Date']).dt.normalize() # Ensure clean match

# 4. Merge & Forward-Fill (The "Anti-Zero" Fix)
merged = pd.merge(prices, sentiment_df, on='Date', how='left')

# If a day has no news, it shouldn't be 0.0 (neutral),
# it should carry over the last known market sentiment.
merged['Sentiment_Score'] = merged['Sentiment_Score'].ffill().fillna(0)
merged['Article_Count'] = merged['Article_Count'].fillna(0)

# 5. Create Lags (Yesterday's News predicts Today's Price)
merged['Prev_Sentiment'] = merged['Sentiment_Score'].shift(1)
merged['Prev_Count'] = merged['Article_Count'].shift(1)

# Final 14-day training window
df_final = merged.dropna()

# 6. Export
df_final.to_csv("amazon_historical_sentiment.csv", index=False)

print(f"Dataframe now has {len(df_final)} days of non-zero features.")
display(df_final)

Fetching data starting from 2026-02-27...
Fetched 100 articles for query: Amazon AND stock
Fetched 98 articles for query: Amazon AND earnings
Fetched 40 articles for query: AMZN stock


[*********************100%***********************]  1 of 1 completed

Dataframe now has 18 days of non-zero features.


,Date,Close,Return,Sentiment_Score,Article_Count,Prev_Sentiment,Prev_Count
1,2026-03-02,208.389999,-0.007667,-0.407400,9,-0.232634,23.0
2,2026-03-03,208.729996,0.001632,0.260242,14,-0.407400,9.0
3,2026-03-04,216.820007,0.038758,0.124617,14,0.260242,14.0
4,2026-03-05,218.940002,0.009778,0.172191,6,0.124617,14.0
5,2026-03-06,213.210007,-0.026172,0.411056,5,0.172191,6.0
6,2026-03-09,213.490005,0.001313,-0.106631,14,0.411056,5.0
7,2026-03-10,214.330002,0.003935,0.196411,19,-0.106631,14.0
8,2026-03-11,212.649994,-0.007838,-0.094701,16,0.196411,19.0
9,2026-03-12,209.529999,-0.014672,0.094783,15,-0.094701,16.0
10,2026-03-13,207.669998,-0.008877,-0.170233,11,0.094783,15.0


In [28]:
# Save the final dataframe to a CSV file
file_name = "amazon_historical_sentiment.csv"

try:
    df_final.to_csv(file_name, index=False)
    print(f"Successfully exported {len(df_final)} rows to '{file_name}'")

    # Optional: Display the first few rows to confirm the format
    display(df_final.head())
except Exception as e:
    print(f"Error saving to CSV: {e}")

Successfully exported 18 rows to 'amazon_historical_sentiment.csv'


,Date,Close,Return,Sentiment_Score,Article_Count,Prev_Sentiment,Prev_Count
1,2026-03-02,208.389999,-0.007667,-0.407400,9,-0.232634,23.0
2,2026-03-03,208.729996,0.001632,0.260242,14,-0.407400,9.0
3,2026-03-04,216.820007,0.038758,0.124617,14,0.260242,14.0
4,2026-03-05,218.940002,0.009778,0.172191,6,0.124617,14.0
5,2026-03-06,213.210007,-0.026172,0.411056,5,0.172191,6.0


In [29]:
# %% [Cell 4: Build & Train TensorFlow Model]
def build_model():
    model = models.Sequential([
        layers.Input(shape=(2,)), # Inputs: Sentiment Score & Article Count
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.1),
        layers.Dense(8, activation='relu'),
        layers.Dense(1, activation='linear') # Linear for regression (predicting % return)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# Prepare Data
X = df_final[['Prev_Sentiment', 'Prev_Count']].values
y = df_final['Return'].values

# Normalize Count
X[:, 1] = X[:, 1] / (X[:, 1].max() + 1)

# Initialize and Train
forecaster = build_model()
print("\nTraining Model...")
history = forecaster.fit(X, y, epochs=50, batch_size=2, verbose=0)
print("Training Complete.")


Training Model...
Training Complete.


In [30]:
# %% [Cell 5: Live Prediction for Tomorrow]
# Fetch the most recent news batch to see "today's mood"
today_news = newsapi.get_everything(q='Amazon AND stock', language='en', sort_by='publishedAt', page_size=20)
today_scores = [get_sentiment(a['title']) for a in today_news['articles']]

if today_scores:
    current_avg = np.mean(today_scores)
    current_count = len(today_scores) / (df_final['Article_Count'].max() + 1)

    pred_input = np.array([[current_avg, current_count]])
    prediction = forecaster.predict(pred_input, verbose=0)[0][0]

    print(f"\n--- PREDICTION FOR NEXT TRADING SESSION ---")
    print(f"Current Sentiment: {current_avg:.4f}")
    print(f"Predicted Return Tomorrow:  {prediction * 100:.4f}%")
    print(f"Action: {'BUY/LONG ' if prediction > 0 else 'SELL/SHORT '}")
else:
    print("Not enough news found for a live prediction.")


--- PREDICTION FOR NEXT TRADING SESSION ---
Current Sentiment: 0.1598
Predicted Return Tomorrow:  -1.5337%
Action: SELL/SHORT 
